# Formula 1 Dataset Analysis

This notebook focuses on exploratory data analysis of the F1 object detection dataset, examining:
- Image characteristics and distributions
- Class balance and object distributions
- Spatial analysis of objects
- Color and lighting conditions

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import yaml
from tqdm import tqdm

## 1. Dataset Overview

First, let's examine the basic characteristics of our dataset.

In [ ]:
# Load data paths and configurations
BASE_DIR = "../data/processed"
IMAGES_DIR = os.path.join(BASE_DIR, "images")
LABELS_DIR = os.path.join(BASE_DIR, "labels")
CONFIG_PATH = os.path.join("../configs/training_configs/training_pipeline.yaml")

# Load class names from config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Count total images
total_images = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))])

print(f"Total images in dataset: {total_images}")

## 2. Image Size Analysis

In [ ]:
def analyze_image_sizes():
    sizes = []
    for img_file in tqdm(os.listdir(IMAGES_DIR)):
        if img_file.endswith(('.jpg', '.png', '.jpeg')):
            img_path = os.path.join(IMAGES_DIR, img_file)
            img = Image.open(img_path)
            sizes.append({
                'width': img.size[0],
                'height': img.size[1],
                'aspect_ratio': img.size[0] / img.size[1]
            })
    
    return pd.DataFrame(sizes)

df_sizes = analyze_image_sizes()

# Plot size distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(data=df_sizes, x='aspect_ratio', bins=30, ax=ax1)
ax1.set_title('Image Aspect Ratio Distribution')

df_sizes.plot.scatter(x='width', y='height', ax=ax2, alpha=0.5)
ax2.set_title('Image Dimensions')

plt.tight_layout()
plt.show()

## 3. Class Distribution Analysis

In [ ]:
def analyze_class_distribution():
    class_counts = Counter()
    objects_per_image = []
    
    for label_file in os.listdir(LABELS_DIR):
        if label_file.endswith('.txt'):
            with open(os.path.join(LABELS_DIR, label_file), 'r') as f:
                labels = f.readlines()
                objects_per_image.append(len(labels))
                for label in labels:
                    class_id = int(label.split()[0])
                    class_counts[class_id] += 1
    
    return class_counts, objects_per_image

class_counts, objects_per_image = analyze_class_distribution()

# Plot class distribution
plt.figure(figsize=(15, 6))
plt.bar(range(len(class_counts)), [class_counts[i] for i in sorted(class_counts.keys())])
plt.title('Class Distribution in Dataset')
plt.xlabel('Class ID')
plt.ylabel('Number of Objects')
plt.show()

# Plot objects per image distribution
plt.figure(figsize=(10, 5))
plt.hist(objects_per_image, bins=range(max(objects_per_image) + 2), align='left')
plt.title('Objects per Image Distribution')
plt.xlabel('Number of Objects')
plt.ylabel('Number of Images')
plt.show()

## 4. Spatial Distribution Analysis

In [ ]:
def analyze_spatial_distribution():
    all_centers = []
    all_sizes = []
    
    for label_file in os.listdir(LABELS_DIR):
        if label_file.endswith('.txt'):
            with open(os.path.join(LABELS_DIR, label_file), 'r') as f:
                for line in f:
                    _, x_center, y_center, width, height = map(float, line.strip().split())
                    all_centers.append([x_center, y_center])
                    all_sizes.append([width, height])
    
    return np.array(all_centers), np.array(all_sizes)

centers, sizes = analyze_spatial_distribution()

# Plot spatial heatmap
plt.figure(figsize=(10, 10))
plt.hist2d(centers[:, 0], centers[:, 1], bins=50, cmap='viridis')
plt.colorbar(label='Number of Objects')
plt.title('Spatial Distribution of Objects')
plt.xlabel('Normalized X Position')
plt.ylabel('Normalized Y Position')
plt.show()

# Plot size distribution
plt.figure(figsize=(10, 5))
plt.scatter(sizes[:, 0], sizes[:, 1], alpha=0.1)
plt.title('Object Size Distribution')
plt.xlabel('Normalized Width')
plt.ylabel('Normalized Height')
plt.show()

## 5. Color Analysis

In [ ]:
def analyze_color_distribution(num_samples=100):
    image_files = os.listdir(IMAGES_DIR)
    sampled_files = np.random.choice(image_files, min(num_samples, len(image_files)), replace=False)
    
    rgb_means = []
    rgb_stds = []
    
    for img_file in sampled_files:
        if img_file.endswith(('.jpg', '.png', '.jpeg')):
            img_path = os.path.join(IMAGES_DIR, img_file)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            rgb_means.append(img.mean(axis=(0, 1)))
            rgb_stds.append(img.std(axis=(0, 1)))
    
    return np.array(rgb_means), np.array(rgb_stds)

rgb_means, rgb_stds = analyze_color_distribution()

# Plot color distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

channels = ['Red', 'Green', 'Blue']
for i, channel in enumerate(channels):
    sns.kdeplot(data=rgb_means[:, i], ax=ax1, label=channel)
ax1.set_title('RGB Channel Mean Distribution')
ax1.legend()

for i, channel in enumerate(channels):
    sns.kdeplot(data=rgb_stds[:, i], ax=ax2, label=channel)
ax2.set_title('RGB Channel Standard Deviation Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Annotation Quality Analysis

In [ ]:
def analyze_annotation_quality():
    box_sizes = []
    box_ratios = []
    
    for label_file in os.listdir(LABELS_DIR):
        if label_file.endswith('.txt'):
            with open(os.path.join(LABELS_DIR, label_file), 'r') as f:
                for line in f:
                    _, _, _, width, height = map(float, line.strip().split())
                    box_sizes.append(width * height)
                    box_ratios.append(width / height)
    
    return box_sizes, box_ratios

box_sizes, box_ratios = analyze_annotation_quality()

# Plot distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(data=box_sizes, bins=50, ax=ax1)
ax1.set_title('Bounding Box Size Distribution')
ax1.set_xlabel('Normalized Area')

sns.histplot(data=box_ratios, bins=50, ax=ax2)
ax2.set_title('Bounding Box Aspect Ratio Distribution')
ax2.set_xlabel('Width/Height Ratio')

plt.tight_layout()
plt.show()